# 20_streamlit - Credit Risk Intelligence Platform

Aplicacao Streamlit para visualizacao e consumo analitico do projeto Credit Risk Intelligence Platform.

**Arquitetura:** Bronze -> Silver -> Gold -> ML -> MLflow -> Prediction -> Streamlit

**Fonte principal:** `credit_risk.analytics.predictions`  
**Modelos:** 6 registrados no MLflow/Unity Catalog  
**Clientes:** 48.744 | **Previsoes:** 292.464 | **Features:** 229

In [0]:
# ============================================================
# 20_STREAMLIT - INSTALACAO DE DEPENDENCIAS
# ============================================================
# Instala streamlit e plotly para execucao da aplicacao
# Em Databricks Apps, as dependencias sao instaladas automaticamente

%pip install streamlit plotly --quiet

In [0]:
# ============================================================
# 20_STREAMLIT - IMPORTS E CONFIGURACAO
# ============================================================

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime

from pyspark.sql import functions as F

# ------------------------------------------------------------
# Configuracao da pagina Streamlit
# Deve ser o primeiro comando do Streamlit
# ------------------------------------------------------------
st.set_page_config(
    page_title="Credit Risk Intelligence",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ------------------------------------------------------------
# Constantes - tabelas fonte (apenas leitura)
# ------------------------------------------------------------
PREDICTIONS_TABLE = "credit_risk.analytics.predictions"
REGISTRY_TABLE = "credit_risk.analytics.mlflow_model_registry"
EVALUATION_TABLE = "credit_risk.analytics.evaluation_results"
AUDIT_PREDICTION_TABLE = "credit_risk.analytics.audit_prediction"

# ------------------------------------------------------------
# Identificacao de execucao
# ------------------------------------------------------------
EXECUTION_ID = "20_streamlit"

In [0]:
# ============================================================
# 20_STREAMLIT - FUNCOES DE CARREGAMENTO DE DADOS
# ============================================================
# Todas as funcoes usam @st.cache_data para evitar recargas
# desnecessarias. Os dados sao carregados uma vez e reutilizados
# em todas as interacoes da aplicacao.
# ============================================================


@st.cache_data(ttl=3600, show_spinner="Carregando previsoes...")
def load_predictions():
    """Carrega a tabela de previsoes e converte para pandas."""
    try:
        df = (spark.table(PREDICTIONS_TABLE)
              .select("SK_ID_CURR", "MODEL_NAME", "MODEL_TYPE", "MODEL_VERSION",
                      "RUN_ID", "PREDICTION", "RISK_PROBABILITY",
                      "RISK_CATEGORY", "RISK_SCORE",
                      "PREDICTION_TIMESTAMP", "EXECUTION_ID")
              )
        pdf = df.toPandas()
        # Garantir tipos corretos
        pdf["PREDICTION"] = pdf["PREDICTION"].astype(int)
        pdf["RISK_PROBABILITY"] = pdf["RISK_PROBABILITY"].astype(float)
        pdf["RISK_SCORE"] = pdf["RISK_SCORE"].astype(float)
        pdf["SK_ID_CURR"] = pdf["SK_ID_CURR"].astype(int)
        return pdf
    except Exception as e:
        st.error(f"Erro ao carregar tabela de previsoes: {str(e)}")
        return pd.DataFrame()


@st.cache_data(ttl=3600, show_spinner="Carregando model registry...")
def load_model_registry():
    """Carrega metadados do MLflow Model Registry."""
    try:
        df = spark.table(REGISTRY_TABLE)
        return df.toPandas()
    except Exception as e:
        st.error(f"Erro ao carregar model registry: {str(e)}")
        return pd.DataFrame()


@st.cache_data(ttl=3600, show_spinner="Carregando avaliacao...")
def load_evaluation():
    """Carrega metricas de avaliacao dos modelos."""
    try:
        df = spark.table(EVALUATION_TABLE)
        return df.toPandas()
    except Exception as e:
        st.error(f"Erro ao carregar avaliacao: {str(e)}")
        return pd.DataFrame()


@st.cache_data(ttl=3600, show_spinner="Carregando auditoria...")
def load_audit():
    """Carrega log de auditoria de predicao."""
    try:
        df = spark.table(AUDIT_PREDICTION_TABLE)
        return df.toPandas()
    except Exception as e:
        st.error(f"Erro ao carregar auditoria: {str(e)}")
        return pd.DataFrame()


def get_customer_predictions(predictions_df, sk_id_curr):
    """Retorna previsoes de um cliente especifico."""
    if predictions_df.empty:
        return pd.DataFrame()
    result = predictions_df[predictions_df["SK_ID_CURR"] == sk_id_curr].copy()
    return result.sort_values("MODEL_TYPE")


def get_risk_distribution(predictions_df, model_filter=None):
    """Retorna distribuicao de risco agregada por categoria."""
    if predictions_df.empty:
        return pd.DataFrame()
    df = predictions_df.copy()
    if model_filter and model_filter != "Todos":
        df = df[df["MODEL_TYPE"] == model_filter]
    dist = df.groupby("RISK_CATEGORY").size().reset_index(name="COUNT")
    total = dist["COUNT"].sum()
    if total > 0:
        dist["PERCENT"] = (dist["COUNT"] / total * 100).round(2)
    else:
        dist["PERCENT"] = 0.0
    # Ordenar por categoria: LOW, MEDIUM, HIGH
    cat_order = {"LOW": 0, "MEDIUM": 1, "HIGH": 2}
    dist["ORDER"] = dist["RISK_CATEGORY"].map(cat_order)
    dist = dist.sort_values("ORDER").drop(columns="ORDER")
    return dist


def get_model_metrics(evaluation_df):
    """Retorna tabela de metricas de avaliacao formatadas."""
    if evaluation_df.empty:
        return pd.DataFrame()
    cols = ["MODEL", "ROC_AUC", "PR_AUC", "PRECISION", "RECALL", "F1",
            "SPECIFICITY", "BALANCED_ACCURACY", "BRIER_SCORE"]
    available = [c for c in cols if c in evaluation_df.columns]
    return evaluation_df[available].copy()

In [0]:
# ============================================================
# 20_STREAMLIT - VALIDACAO DE QUALIDADE DE DADOS
# ============================================================
# Verifica integridade dos dados antes de exibir na interface.
# Nao corrige silenciosamente; apenas alerta o usuario.
# ============================================================


def validate_predictions_data(predictions_df):
    """Valida integridade dos dados de previsoes."""
    if predictions_df.empty:
        return {"valid": False, "warnings": ["Tabela de previsoes vazia ou indisponivel."]}

    warnings = []

    # Validar RISK_PROBABILITY no intervalo [0, 1]
    prob_min = predictions_df["RISK_PROBABILITY"].min()
    prob_max = predictions_df["RISK_PROBABILITY"].max()
    if prob_min < 0:
        warnings.append(f"RISK_PROBABILITY contem valor minimo invalido: {prob_min:.4f} (esperado >= 0)")
    if prob_max > 1:
        warnings.append(f"RISK_PROBABILITY contem valor maximo invalido: {prob_max:.4f} (esperado <= 1)")

    # Validar RISK_SCORE no intervalo [0, 100]
    score_min = predictions_df["RISK_SCORE"].min()
    score_max = predictions_df["RISK_SCORE"].max()
    if score_min < 0:
        warnings.append(f"RISK_SCORE contem valor minimo invalido: {score_min:.2f} (esperado >= 0)")
    if score_max > 100:
        warnings.append(f"RISK_SCORE contem valor maximo invalido: {score_max:.2f} (esperado <= 100)")

    # Validar PREDICTION em {0, 1}
    unique_preds = set(predictions_df["PREDICTION"].unique())
    if not unique_preds.issubset({0, 1}):
        warnings.append(f"PREDICTION contem valores invalidos: {unique_preds - {0, 1}}")

    # Validar RISK_CATEGORY
    valid_cats = {"LOW", "MEDIUM", "HIGH"}
    actual_cats = set(predictions_df["RISK_CATEGORY"].unique())
    if not actual_cats.issubset(valid_cats):
        warnings.append(f"RISK_CATEGORY contem valores invalidos: {actual_cats - valid_cats}")

    return {"valid": len(warnings) == 0, "warnings": warnings}


def show_dq_warning(validation_result):
    """Exibe alerta de data quality se houver inconsistencias."""
    if not validation_result["valid"]:
        st.warning("Data quality warning: Foram detectadas inconsistencias nos dados.")
        with st.expander("Detalhes da validacao"):
            for w in validation_result["warnings"]:
                st.write(f"- {w}")

In [0]:
# ============================================================
# 20_STREAMLIT - CARREGAMENTO DE DADOS E SIDEBAR
# ============================================================

# ------------------------------------------------------------
# Carregamento de dados (cacheado via @st.cache_data)
# ------------------------------------------------------------
predictions_df = load_predictions()
registry_df = load_model_registry()
evaluation_df = load_evaluation()
audit_df = load_audit()

# ------------------------------------------------------------
# Validacao de data quality
# ------------------------------------------------------------
dq_result = validate_predictions_data(predictions_df)

# ------------------------------------------------------------
# Sidebar - Navegacao
# ------------------------------------------------------------
st.sidebar.title("Navegacao")

PAGES = [
    "Visao Executiva",
    "Analise de Cliente",
    "Comparacao de Modelos",
    "Distribuicao de Risco",
    "Auditoria",
    "Sobre o Projeto"
]

page = st.sidebar.selectbox("Selecione a pagina", PAGES)

# ------------------------------------------------------------
# Sidebar - Filtros globais
# ------------------------------------------------------------
st.sidebar.divider()
st.sidebar.subheader("Filtros globais")

if not predictions_df.empty:
    model_list = sorted(predictions_df["MODEL_TYPE"].unique().tolist())
else:
    model_list = []

selected_model = st.sidebar.selectbox("Modelo", ["Todos"] + model_list)
selected_risk = st.sidebar.selectbox("Categoria de risco", ["Todas", "LOW", "MEDIUM", "HIGH"])
selected_prediction = st.sidebar.selectbox("Predicao", ["Todas", "0 (Baixo risco)", "1 (Alto risco)"])

# ------------------------------------------------------------
# Botao de atualizacao de dados
# ------------------------------------------------------------
st.sidebar.divider()
if st.sidebar.button("Atualizar dados"):
    st.cache_data.clear()
    st.rerun()

st.sidebar.markdown("---")
st.sidebar.markdown(f"**Ultima atualizacao:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ------------------------------------------------------------
# Aplicacao de filtros globais
# ------------------------------------------------------------
filtered_df = predictions_df.copy()

if selected_model != "Todos":
    filtered_df = filtered_df[filtered_df["MODEL_TYPE"] == selected_model]

if selected_risk != "Todas":
    filtered_df = filtered_df[filtered_df["RISK_CATEGORY"] == selected_risk]

if selected_prediction != "Todas":
    pred_val = 1 if "1" in selected_prediction else 0
    filtered_df = filtered_df[filtered_df["PREDICTION"] == pred_val]

# ------------------------------------------------------------
# Titulo da aplicacao
# ------------------------------------------------------------
st.title("Credit Risk Intelligence Platform")
st.markdown("Plataforma analitica para avaliacao e monitoramento de risco de credito baseada em Machine Learning")
st.markdown("Pipeline completo desenvolvido em Databricks utilizando arquitetura Medallion, Delta Lake, Machine Learning, MLflow e Streamlit.")
st.divider()

# Exibir alerta de DQ se necessario
show_dq_warning(dq_result)

In [0]:
# ============================================================
# 20_STREAMLIT - VISAO EXECUTIVA
# ============================================================

if page == "Visao Executiva":
    st.header("Visao Executiva")

    if filtered_df.empty:
        st.warning("Nenhum dado disponivel com os filtros selecionados.")
    else:
        # --------------------------------------------------------
        # KPIs principais
        # --------------------------------------------------------
        unique_customers = int(filtered_df["SK_ID_CURR"].nunique())
        unique_models = int(filtered_df["MODEL_TYPE"].nunique())
        total_predictions = int(len(filtered_df))
        avg_probability = float(filtered_df["RISK_PROBABILITY"].mean())

        # Contagem por categoria de risco
        high_count = int(len(filtered_df[filtered_df["RISK_CATEGORY"] == "HIGH"]))
        medium_count = int(len(filtered_df[filtered_df["RISK_CATEGORY"] == "MEDIUM"]))
        low_count = int(len(filtered_df[filtered_df["RISK_CATEGORY"] == "LOW"]))

        # Exibir KPIs em colunas
        col1, col2, col3, col4 = st.columns(4)
        col1.metric("Clientes analisados", f"{unique_customers:,}")
        col2.metric("Modelos registrados", f"{unique_models}")
        col3.metric("Total de previsoes", f"{total_predictions:,}")
        col4.metric("Probabilidade media", f"{avg_probability:.4f}")

        col5, col6, col7 = st.columns(3)
        col5.metric("Previsoes LOW", f"{low_count:,}")
        col6.metric("Previsoes MEDIUM", f"{medium_count:,}")
        col7.metric("Previsoes HIGH", f"{high_count:,}")

        st.divider()

        # --------------------------------------------------------
        # Grafico de distribuicao de risco
        # --------------------------------------------------------
        st.subheader("Distribuicao de risco")
        risk_dist = get_risk_distribution(filtered_df)

        if not risk_dist.empty:
            col_chart, col_data = st.columns([3, 2])

            with col_chart:
                fig = px.bar(
                    risk_dist,
                    x="RISK_CATEGORY",
                    y="COUNT",
                    text="COUNT",
                    labels={
                        "RISK_CATEGORY": "Categoria de risco",
                        "COUNT": "Quantidade"
                    },
                    title="Distribuicao por categoria de risco"
                )
                fig.update_traces(textposition="outside")
                st.plotly_chart(fig, use_container_width=True)

            with col_data:
                st.dataframe(
                    risk_dist[["RISK_CATEGORY", "COUNT", "PERCENT"]].rename(columns={
                        "RISK_CATEGORY": "Categoria",
                        "COUNT": "Quantidade",
                        "PERCENT": "Percentual (%)"
                    }),
                    use_container_width=True,
                    hide_index=True
                )

        st.divider()

        # --------------------------------------------------------
        # Documentacao da classificacao operacional
        # --------------------------------------------------------
        with st.expander("Classificacao operacional utilizada"):
            st.markdown("""
            **LOW:** probabilidade < 10%
            
            **MEDIUM:** 10% <= probabilidade < 30%
            
            **HIGH:** probabilidade >= 30%

            As faixas apresentadas sao classificacoes operacionais para analise.
            Nao representam, isoladamente, uma decisao de concessao ou recusa de credito.
            """)

In [0]:
# ============================================================
# 20_STREAMLIT - ANALISE DE CLIENTE
# ============================================================

if page == "Analise de Cliente":
    st.header("Analise de Cliente")

    if predictions_df.empty:
        st.warning("Dados de previsoes indisponiveis.")
    else:
        # --------------------------------------------------------
        # Selecao de cliente
        # --------------------------------------------------------
        min_id = int(predictions_df["SK_ID_CURR"].min())
        max_id = int(predictions_df["SK_ID_CURR"].max())

        col_input, col_hint = st.columns([1, 2])
        with col_input:
            selected_customer = st.number_input(
                "Digite o SK_ID_CURR do cliente",
                min_value=min_id,
                max_value=max_id,
                value=min_id,
                step=1
            )
        with col_hint:
            st.caption(f"Intervalo disponivel: {min_id} a {max_id} | Total de clientes: {predictions_df['SK_ID_CURR'].nunique():,}")

        # Buscar previsoes do cliente
        customer_preds = get_customer_predictions(predictions_df, int(selected_customer))

        if customer_preds.empty:
            st.warning(f"Cliente {selected_customer} nao encontrado na tabela de previsoes.")
        else:
            # ----------------------------------------------------
            # Card do cliente (modelo LightGBM como referencia)
            # ----------------------------------------------------
            lgbm_pred = customer_preds[customer_preds["MODEL_TYPE"] == "LightGBM"]
            ref_pred = lgbm_pred.iloc[0] if not lgbm_pred.empty else customer_preds.iloc[0]

            st.subheader(f"Cliente {int(selected_customer)}")

            col1, col2, col3, col4, col5 = st.columns(5)
            col1.metric("Cliente", f"{int(selected_customer)}")
            col2.metric("Modelo", f"{ref_pred['MODEL_TYPE']}")
            col3.metric("Versao", f"{ref_pred['MODEL_VERSION']}")
            col4.metric("Probabilidade", f"{ref_pred['RISK_PROBABILITY']*100:.2f}%")
            col5.metric("Categoria", f"{ref_pred['RISK_CATEGORY']}")

            col6, col7 = st.columns(2)
            col6.metric("Score", f"{ref_pred['RISK_SCORE']:.2f}")
            col7.metric("Predicao", f"{int(ref_pred['PREDICTION'])}")

            st.markdown("---")
            st.markdown(f"**Probabilidade estimada pelo modelo:** {ref_pred['RISK_PROBABILITY']:.4f} ({ref_pred['RISK_PROBABILITY']*100:.2f}%)")
            st.markdown(f"**Classificacao operacional do modelo:** {ref_pred['RISK_CATEGORY']}")

            st.divider()

            # ----------------------------------------------------
            # Tabela de previsoes por modelo
            # ----------------------------------------------------
            st.subheader("Previsoes por modelo")
            display_cols = ["MODEL_TYPE", "MODEL_VERSION", "PREDICTION",
                            "RISK_PROBABILITY", "RISK_SCORE", "RISK_CATEGORY"]
            display_df = customer_preds[display_cols].rename(columns={
                "MODEL_TYPE": "Modelo",
                "MODEL_VERSION": "Versao",
                "PREDICTION": "Predicao",
                "RISK_PROBABILITY": "Probabilidade",
                "RISK_SCORE": "Score",
                "RISK_CATEGORY": "Categoria"
            })
            st.dataframe(display_df, use_container_width=True, hide_index=True)

            st.divider()

            # ----------------------------------------------------
            # Grafico de comparacao de modelos
            # ----------------------------------------------------
            st.subheader("Comparacao de modelos")

            fig = px.bar(
                customer_preds,
                x="MODEL_TYPE",
                y="RISK_PROBABILITY",
                text=customer_preds["RISK_PROBABILITY"].apply(lambda x: f"{x:.4f}"),
                labels={
                    "MODEL_TYPE": "Modelo",
                    "RISK_PROBABILITY": "Probabilidade de risco"
                },
                title="Probabilidade de inadimplencia por modelo"
            )
            fig.update_traces(textposition="outside")
            st.plotly_chart(fig, use_container_width=True)

            # Estatisticas das probabilidades
            prob_mean = float(customer_preds["RISK_PROBABILITY"].mean())
            prob_min = float(customer_preds["RISK_PROBABILITY"].min())
            prob_max = float(customer_preds["RISK_PROBABILITY"].max())
            prob_std = float(customer_preds["RISK_PROBABILITY"].std())

            col1, col2, col3, col4 = st.columns(4)
            col1.metric("Probabilidade media", f"{prob_mean:.4f}")
            col2.metric("Probabilidade minima", f"{prob_min:.4f}")
            col3.metric("Probabilidade maxima", f"{prob_max:.4f}")
            col4.metric("Desvio padrao", f"{prob_std:.4f}")

            st.divider()

            # Documentacao da classificacao
            with st.expander("Classificacao operacional utilizada"):
                st.markdown("""
                **LOW:** probabilidade < 10%
                
                **MEDIUM:** 10% <= probabilidade < 30%
                
                **HIGH:** probabilidade >= 30%

                As faixas apresentadas sao classificacoes operacionais para analise.
                Nao representam, isoladamente, uma decisao de concessao ou recusa de credito.
                """)

In [0]:
# ============================================================
# 20_STREAMLIT - COMPARACAO DE MODELOS
# ============================================================

if page == "Comparacao de Modelos":
    st.header("Comparacao de Modelos")

    if evaluation_df.empty:
        st.warning("Dados de avaliacao indisponiveis.")
    else:
        st.markdown("Metricas de avaliacao dos modelos em conjunto de validacao holdout estratificado (20% de ml_train).")

        # Tabela de metricas
        metrics_df = get_model_metrics(evaluation_df)

        display_df = metrics_df.rename(columns={
            "MODEL": "Modelo",
            "ROC_AUC": "ROC-AUC",
            "PR_AUC": "PR-AUC",
            "PRECISION": "Precision",
            "RECALL": "Recall",
            "F1": "F1",
            "SPECIFICITY": "Specificity",
            "BALANCED_ACCURACY": "Balanced Accuracy",
            "BRIER_SCORE": "Brier Score"
        })

        st.dataframe(display_df, use_container_width=True, hide_index=True)

        st.divider()

        # Graficos de metricas individuais
        st.subheader("Visualizacao de metricas")

        metrics_to_plot = [
            ("ROC_AUC", "ROC-AUC"),
            ("PR_AUC", "PR-AUC"),
            ("RECALL", "Recall"),
            ("PRECISION", "Precision"),
            ("F1", "F1")
        ]

        for metric_col, metric_label in metrics_to_plot:
            if metric_col in evaluation_df.columns:
                fig = px.bar(
                    evaluation_df,
                    x="MODEL",
                    y=metric_col,
                    text=evaluation_df[metric_col].apply(lambda x: f"{x:.4f}"),
                    labels={"MODEL": "Modelo", metric_col: metric_label},
                    title=f"{metric_label} por modelo"
                )
                fig.update_traces(textposition="outside")
                st.plotly_chart(fig, use_container_width=True)

In [0]:
# ============================================================
# 20_STREAMLIT - DISTRIBUICAO DE RISCO
# ============================================================

if page == "Distribuicao de Risco":
    st.header("Distribuicao de Risco")

    if predictions_df.empty:
        st.warning("Dados de previsoes indisponiveis.")
    else:
        # Distribuicao por modelo (com seletor)
        st.subheader("Distribuicao por modelo")

        model_list = sorted(predictions_df["MODEL_TYPE"].unique().tolist())
        selected_model_dist = st.selectbox("Selecione o modelo", ["Todos"] + model_list)

        dist_df = predictions_df.copy()
        if selected_model_dist != "Todos":
            dist_df = dist_df[dist_df["MODEL_TYPE"] == selected_model_dist]

        if not dist_df.empty:
            total_customers = int(dist_df["SK_ID_CURR"].nunique())
            total_preds = int(len(dist_df))
            avg_prob = float(dist_df["RISK_PROBABILITY"].mean())

            low_pct = len(dist_df[dist_df["RISK_CATEGORY"] == "LOW"]) / total_preds * 100 if total_preds > 0 else 0
            med_pct = len(dist_df[dist_df["RISK_CATEGORY"] == "MEDIUM"]) / total_preds * 100 if total_preds > 0 else 0
            high_pct = len(dist_df[dist_df["RISK_CATEGORY"] == "HIGH"]) / total_preds * 100 if total_preds > 0 else 0

            col1, col2, col3, col4, col5 = st.columns(5)
            col1.metric("Total de clientes", f"{total_customers:,}")
            col2.metric("Risco baixo (%)", f"{low_pct:.1f}%")
            col3.metric("Risco medio (%)", f"{med_pct:.1f}%")
            col4.metric("Risco alto (%)", f"{high_pct:.1f}%")
            col5.metric("Probabilidade media", f"{avg_prob:.4f}")

            st.divider()

            risk_dist = get_risk_distribution(dist_df)
            if not risk_dist.empty:
                fig = px.bar(
                    risk_dist,
                    x="RISK_CATEGORY",
                    y="COUNT",
                    text="COUNT",
                    color="RISK_CATEGORY",
                    labels={"RISK_CATEGORY": "Categoria de risco", "COUNT": "Quantidade"},
                    title="Distribuicao por categoria de risco"
                )
                fig.update_traces(textposition="outside")
                st.plotly_chart(fig, use_container_width=True)

        st.divider()

        # Distribuicao por modelo e categoria (grouped bar)
        st.subheader("Distribuicao por modelo e categoria")

        model_cat = (predictions_df.groupby(["MODEL_TYPE", "RISK_CATEGORY"]).size().reset_index(name="COUNT"))

        if not model_cat.empty:
            fig = px.bar(
                model_cat,
                x="MODEL_TYPE",
                y="COUNT",
                color="RISK_CATEGORY",
                barmode="group",
                labels={"MODEL_TYPE": "Modelo", "COUNT": "Quantidade", "RISK_CATEGORY": "Categoria"},
                title="Distribuicao de risco por modelo"
            )
            st.plotly_chart(fig, use_container_width=True)

        st.divider()

        # Histograma de probabilidade
        st.subheader("Distribuicao da probabilidade de risco")

        hist_model = st.selectbox("Filtrar por modelo (histograma)", ["Todos"] + model_list, key="hist_model")

        hist_df = predictions_df.copy()
        if hist_model != "Todos":
            hist_df = hist_df[hist_df["MODEL_TYPE"] == hist_model]

        if not hist_df.empty:
            fig = px.histogram(
                hist_df,
                x="RISK_PROBABILITY",
                nbins=50,
                labels={"RISK_PROBABILITY": "Probabilidade de risco"},
                title="Histograma de probabilidade de inadimplencia"
            )
            st.plotly_chart(fig, use_container_width=True)

            prob_min = float(hist_df["RISK_PROBABILITY"].min())
            prob_mean = float(hist_df["RISK_PROBABILITY"].mean())
            prob_median = float(hist_df["RISK_PROBABILITY"].median())
            prob_max = float(hist_df["RISK_PROBABILITY"].max())

            col1, col2, col3, col4 = st.columns(4)
            col1.metric("Minimo", f"{prob_min:.4f}")
            col2.metric("Media", f"{prob_mean:.4f}")
            col3.metric("Mediana", f"{prob_median:.4f}")
            col4.metric("Maximo", f"{prob_max:.4f}")

In [0]:
# ============================================================
# 20_STREAMLIT - AUDITORIA
# ============================================================

if page == "Auditoria":
    st.header("Auditoria")

    # Registro de execucao de predicao
    st.subheader("Registro de execucao de predicao")

    if audit_df.empty:
        st.warning("Dados de auditoria indisponiveis.")
    else:
        audit_display = audit_df.rename(columns={
            "execution_id": "Execution ID",
            "execution_timestamp": "Timestamp",
            "models_processed": "Modelos processados",
            "models_failed": "Modelos com falha",
            "total_predictions": "Total de previsoes",
            "test_rows": "Linhas de teste",
            "feature_count": "Features",
            "status": "Status",
            "execution_time_seconds": "Tempo de execucao (s)",
            "notes": "Observacoes"
        })
        st.dataframe(audit_display, use_container_width=True, hide_index=True)

    st.divider()

    # Model Registry
    st.subheader("Model Registry")

    if registry_df.empty:
        st.warning("Dados do model registry indisponiveis.")
    else:
        registry_display = registry_df[[
            "model_name", "model_version", "run_id", "model_uri",
            "roc_auc", "pr_auc", "status"
        ]].rename(columns={
            "model_name": "Modelo",
            "model_version": "Versao",
            "run_id": "Run ID",
            "model_uri": "Model URI",
            "roc_auc": "ROC-AUC",
            "pr_auc": "PR-AUC",
            "status": "Status"
        })
        st.dataframe(registry_display, use_container_width=True, hide_index=True)

        st.divider()

        with st.expander("Detalhes tecnicos do registry"):
            full_registry = registry_df.rename(columns={
                "execution_id": "Execution ID",
                "model_name": "Modelo",
                "model_type": "Tipo",
                "run_id": "Run ID",
                "experiment_id": "Experiment ID",
                "model_version": "Versao",
                "model_uri": "Model URI",
                "training_rows": "Linhas de treino",
                "validation_rows": "Linhas de validacao",
                "feature_count": "Features",
                "roc_auc": "ROC-AUC",
                "pr_auc": "PR-AUC",
                "precision": "Precision",
                "recall": "Recall",
                "f1": "F1",
                "specificity": "Specificity",
                "balanced_accuracy": "Balanced Accuracy",
                "brier_score": "Brier Score",
                "status": "Status",
                "created_at": "Criado em"
            })
            st.dataframe(full_registry, use_container_width=True, hide_index=True)

In [0]:
# ============================================================
# 20_STREAMLIT - SOBRE O PROJETO
# ============================================================

if page == "Sobre o Projeto":
    st.header("Sobre o Projeto")

    st.markdown("O **Credit Risk Intelligence Platform** e uma plataforma analitica completa para avaliacao e monitoramento de risco de credito, baseada em Machine Learning e desenvolvida em Databricks.")

    st.divider()

    # Arquitetura do pipeline
    st.subheader("Arquitetura do Pipeline")

    st.markdown("""
    ```
    Bronze
      |
      v
    Silver
      |
      v
    Gold
      |
      v
    ML Dataset
      |
      v
    EDA
      |
      v
    Training
      |
      v
    Evaluation
      |
      v
    MLflow
      |
      v
    Prediction
      |
      v
    Streamlit
    ```
    """)

    st.divider()

    # Arquitetura Medallion
    st.subheader("Arquitetura Medallion")

    col1, col2, col3 = st.columns(3)

    with col1:
        st.markdown("**BRONZE**")
        st.markdown("Dados brutos e auditoria")
        st.caption("8 tabelas originais do dataset Home Credit Default Risk")

    with col2:
        st.markdown("**SILVER**")
        st.markdown("Qualidade, limpeza e padronizacao")
        st.caption("8 tabelas com colunas de DQ e controle")

    with col3:
        st.markdown("**GOLD**")
        st.markdown("Features e dataset para Machine Learning")
        st.caption("229 features + SK_ID_CURR + TARGET")

    col4, col5, col6 = st.columns(3)

    with col4:
        st.markdown("**MLFLOW**")
        st.markdown("Experimentacao e Model Registry")
        st.caption("6 modelos registrados no Unity Catalog")

    with col5:
        st.markdown("**PREDICTION**")
        st.markdown("Inferencia e classificacao de risco")
        st.caption("292.464 previsoes sobre 48.744 clientes")

    with col6:
        st.markdown("**STREAMLIT**")
        st.markdown("Visualizacao e consumo analitico")
        st.caption("Dashboard interativo de risco de credito")

    st.divider()

    # Tecnologias utilizadas
    st.subheader("Tecnologias utilizadas")

    techs = [
        "Databricks", "Delta Lake", "Apache Spark", "PySpark", "Python", "SQL",
        "Machine Learning", "LightGBM", "XGBoost", "Scikit-learn",
        "MLflow", "Unity Catalog", "Streamlit", "Plotly"
    ]

    tech_cols = st.columns(7)
    for i, tech in enumerate(techs):
        tech_cols[i % 7].markdown(f"- {tech}")

    st.divider()

    # Classificacao operacional
    st.subheader("Classificacao operacional utilizada")

    st.markdown("""
    | Faixa | Condicao |
    | --- | --- |
    | LOW | probabilidade < 10% |
    | MEDIUM | 10% <= probabilidade < 30% |
    | HIGH | probabilidade >= 30% |
    """)

    st.markdown("As faixas apresentadas sao classificacoes operacionais para analise. Nao representam, isoladamente, uma decisao de concessao ou recusa de credito.")

    st.divider()

    # Informacoes do dataset
    st.subheader("Informacoes do dataset")

    st.markdown("""
    | Item | Valor |
    | --- | --- |
    | Dataset | Home Credit Default Risk |
    | Clientes analisados | 48.744 |
    | Features | 229 |
    | Modelos | 6 |
    | Total de previsoes | 292.464 |
    | Registry | credit_risk.analytics.mlflow_model_registry |
    | Tabela de previsoes | credit_risk.analytics.predictions |
    """)

    st.divider()

    # Resumo da aplicacao
    st.subheader("Resumo da Aplicacao")
    st.markdown("""
    **Application:** Credit Risk Intelligence Platform  
    **Data source:** credit_risk.analytics.predictions  
    **Customers:** 48,744  
    **Models:** 6  
    **Features:** 229  
    **Predictions:** 292,464  
    **MLflow:** Unity Catalog  
    **Registry:** credit_risk.analytics.mlflow_model_registry  
    **Application status:** READY
    """)

In [0]:
# ============================================================
# 20_STREAMLIT - RESUMO FINAL
# ============================================================

print("=" * 40)
print("20_STREAMLIT - SUMMARY")
print("=" * 40)
print()
print("Application:")
print("Credit Risk Intelligence Platform")
print()
print("Data source:")
print("credit_risk.analytics.predictions")
print()
print("Customers:")
print("48,744")
print()
print("Models:")
print("6")
print()
print("Features:")
print("229")
print()
print("Predictions:")
print("292,464")
print()
print("MLflow:")
print("Unity Catalog")
print()
print("Registry:")
print("credit_risk.analytics.mlflow_model_registry")
print()
print("Application status:")
print("READY")
print()
print("=" * 40)